In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path

# Main project directory
PROJECT_DIR = Path("/content/drive/MyDrive/deeplearning")

# Dataset directory
DATASET_DIR = PROJECT_DIR / "Dataset" / "Dataset_for_yolo"

# Project output directories
NOTEBOOKS_DIR = PROJECT_DIR / "notebooks"
MODELS_DIR = PROJECT_DIR / "models"
OUTPUTS_DIR = PROJECT_DIR / "outputs"
RESULTS_DIR = PROJECT_DIR / "results"

for directory in [
    NOTEBOOKS_DIR,
    MODELS_DIR,
    OUTPUTS_DIR,
    RESULTS_DIR,
    RESULTS_DIR / "metrics",
    RESULTS_DIR / "plots",
    OUTPUTS_DIR / "detections",
    OUTPUTS_DIR / "masks",
    OUTPUTS_DIR / "visualizations",
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Dataset directory:", DATASET_DIR)
print("Dataset exists:", DATASET_DIR.exists())

Project directory: /content/drive/MyDrive/deeplearning
Dataset directory: /content/drive/MyDrive/deeplearning/Dataset/Dataset_for_yolo
Dataset exists: True


In [3]:
if not DATASET_DIR.exists():
    raise FileNotFoundError(
        f"Dataset directory not found: {DATASET_DIR}"
    )

print("Dataset directory contents:\n")

for item in sorted(DATASET_DIR.iterdir()):
    item_type = "Directory" if item.is_dir() else "File"
    print(f"{item_type}: {item.name}")

Dataset directory contents:

File: README.dataset.txt
File: README.roboflow.txt
File: data.yaml
Directory: test
Directory: train
Directory: valid


In [4]:
yaml_files = (
    list(DATASET_DIR.rglob("data.yaml"))
    + list(DATASET_DIR.rglob("data.yml"))
)

yaml_files = sorted(set(yaml_files))

print("Configuration files found:", len(yaml_files))

for index, path in enumerate(yaml_files, start=1):
    print(f"{index}. {path}")

Configuration files found: 1
1. /content/drive/MyDrive/deeplearning/Dataset/Dataset_for_yolo/data.yaml


In [5]:
import yaml

if not yaml_files:
    raise FileNotFoundError(
        "No data.yaml or data.yml found. "
        "Check whether the Roboflow dataset was extracted correctly."
    )

# Prefer a configuration directly inside DATASET_DIR.
direct_yaml = [
    path for path in yaml_files
    if path.parent == DATASET_DIR
]

DATA_YAML = direct_yaml[0] if direct_yaml else yaml_files[0]

with open(DATA_YAML, "r") as file:
    dataset_config = yaml.safe_load(file)

print("Using configuration:", DATA_YAML)
print("\nDataset configuration:")

for key, value in dataset_config.items():
    print(f"{key}: {value}")

Using configuration: /content/drive/MyDrive/deeplearning/Dataset/Dataset_for_yolo/data.yaml

Dataset configuration:
train: ../train/images
val: ../valid/images
test: ../test/images
nc: 1
names: ['bottle']
roboflow: {'workspace': 'waste-detection-zxduh', 'project': 'flow-9dt9s-ysldg', 'version': 1, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/waste-detection-zxduh/flow-9dt9s-ysldg/dataset/1'}


In [6]:
names = dataset_config.get("names", {})
nc = dataset_config.get("nc")

if isinstance(names, dict):
    class_names = [
        names[key] for key in sorted(names, key=lambda x: int(x))
    ]
elif isinstance(names, list):
    class_names = names
else:
    class_names = []

if nc is not None and class_names and int(nc) != len(class_names):
    raise ValueError(
        f"Class count mismatch: nc={nc}, "
        f"but {len(class_names)} class names were found."
    )

if not class_names:
    raise ValueError(
        "Class names are missing or invalid in the YAML file."
    )

print("Number of classes:", len(class_names))
print("Classes:")

for class_id, name in enumerate(class_names):
    print(f"{class_id}: {name}")

Number of classes: 1
Classes:
0: bottle


In [7]:
def resolve_dataset_path(path_value):
    if not path_value:
        return None

    path = Path(path_value)

    if path.is_absolute():
        return path

    yaml_base = DATA_YAML.parent
    configured_root = dataset_config.get("path", ".")

    root = Path(configured_root)

    if not root.is_absolute():
        root = yaml_base / root

    # First try the path relative to the configured dataset root.
    candidate = root / path

    if candidate.exists():
        return candidate.resolve()

    # Then try relative to the directory containing data.yaml.
    candidate = yaml_base / path

    return candidate.resolve()


split_paths = {}

for split in ["train", "val", "valid", "test"]:
    if split not in dataset_config:
        continue

    resolved = resolve_dataset_path(dataset_config[split])
    split_paths[split] = resolved

    print(f"{split}: {resolved}")
    print("  Exists:", resolved.exists() if resolved else False)

train: /content/drive/MyDrive/deeplearning/Dataset/train/images
  Exists: False
val: /content/drive/MyDrive/deeplearning/Dataset/valid/images
  Exists: False
test: /content/drive/MyDrive/deeplearning/Dataset/test/images
  Exists: False


In [8]:
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".webp", ".tif", ".tiff"
}

image_counts = {}

for split, split_dir in split_paths.items():
    if split_dir is None or not split_dir.exists():
        image_counts[split] = 0
        print(f"{split}: Directory not found")
        continue

    images = [
        p for p in split_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    image_counts[split] = len(images)

    print(f"{split}: {len(images)} images")

train: Directory not found
val: Directory not found
test: Directory not found


In [9]:
for split, image_dir in split_paths.items():
    if image_dir is None or not image_dir.exists():
        continue

    # Standard Roboflow layout: train/images and train/labels
    label_dir = image_dir.parent / "labels"

    print(f"\nSplit: {split}")
    print("Image directory:", image_dir)
    print("Expected label directory:", label_dir)
    print("Label directory exists:", label_dir.exists())

    if label_dir.exists():
        label_files = list(label_dir.rglob("*.txt"))
        print("Label files:", len(label_files))

        if label_files:
            print("Example label:", label_files[0].name)
            print(label_files[0].read_text()[:500])

In [10]:
import json

setup_report = {
    "project_name": "Plastic Waste Detection and Segmentation",
    "detector": "YOLOv8m",
    "segmenter": "SAM 2.1 Small",
    "project_directory": str(PROJECT_DIR),
    "dataset_directory": str(DATASET_DIR),
    "data_yaml": str(DATA_YAML),
    "class_names": class_names,
    "number_of_classes": len(class_names),
    "image_counts": image_counts,
    "split_paths": {
        key: str(value) if value else None
        for key, value in split_paths.items()
    },
}

report_path = RESULTS_DIR / "dataset_setup_report.json"

with open(report_path, "w") as file:
    json.dump(setup_report, file, indent=4)

print("Setup report saved to:", report_path)

Setup report saved to: /content/drive/MyDrive/deeplearning/results/dataset_setup_report.json
